# Modeling IHSG — Dual-Branch Multivariate LSTM (v3)
## Analisis Prediksi IHSG Menggunakan Multivariate LSTM Berdasarkan Data Historis dan Variabel Makroekonomi Periode 2015–2024

**Pendekatan v3 — Single Run, Hyperparameter Berdasarkan Observasi Empiris**

Berbeda dari v1/v2 yang menggunakan grid search proxy (5-epoch) untuk mencari
hyperparameter arsitektur, v3 **tidak menggunakan grid search**. Hyperparameter
arsitektur dikunci berdasarkan observasi empiris dari seluruh eksperimen
sebelumnya (v1 & v2):

| Hyperparameter | Nilai | Dasar Empiris |
|---|---|---|
| `seq_daily` | 90 | Konsisten menjadi best config univariate di v1 & v2; horizon ~4.5 bulan trading |
| `lstm_units` | 128 | Konsisten lebih baik dari 64 di seluruh run sebelumnya |
| `dropout_rate` | 0.2 | Standar (Srivastava et al., 2014); tidak ada indikasi perlu diubah dari eksperimen |
| `learning_rate` | 0.0001 | Saat retrain manual di v1 dengan lr=0.0001, val_loss jauh lebih stabil dibanding lr=0.0005/0.001 |
| `batch_size` | 32 | Aman terhadap memory crash (OOM) yang terjadi pada grid search v2 |

**Pipeline:**
1. Load artefak dari `data/processed_v2/` (sudah terverifikasi, y_test canonical)
2. Definisi arsitektur model
3. Ablation study `seq_monthly` ∈ {3, 6, 12} untuk multivariate — satu-satunya
   parameter yang belum terjawab secara empiris
4. Final training **single run** (bukan multi-seed) untuk multivariate & univariate
5. Evaluasi test set: RMSE, MAPE, R²
6. Simpan artefak ke `model/model_multivariate_v3/`

**Perbedaan dari v2:**
- Tidak ada grid search proxy sama sekali (dihapus karena 5-epoch proxy
  terbukti tidak reliable, dan 15-epoch proxy masih proxy dengan masalah yang sama)
- Hyperparameter arsitektur dikunci dari observasi empiris eksperimen sebelumnya
- Final training single run (bukan multi-seed) dengan patience lebih besar
  (ES=30, LR=15) untuk memberi model waktu eksplorasi lebih lama sebelum berhenti
- seed=42 digunakan untuk reproducibility


## Section 1 — Imports & Konfigurasi

In [1]:
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from pathlib import Path
from datetime import datetime
from sklearn.metrics import mean_squared_error, r2_score

from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU') or 'None (CPU only)'}")


TensorFlow  : 2.20.0
NumPy       : 2.2.6
Pandas      : 2.3.3
GPU devices : None (CPU only)


In [2]:
ROOT           = Path(os.getcwd()).parent
DATA_PROCESSED = ROOT / "data" / "processed_v2"        # ← pakai v2 (sudah terverifikasi)
MODEL_DIR      = ROOT / "model" / "model_multivariate_v3"  # ← output v3
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT          :", ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED, "(reused dari v2)")
print("MODEL_DIR     :", MODEL_DIR, "(baru)")
assert DATA_PROCESSED.exists(), f"Folder tidak ditemukan: {DATA_PROCESSED}"


ROOT          : d:\project_skripsi
DATA_PROCESSED: d:\project_skripsi\data\processed_v2 (reused dari v2)
MODEL_DIR     : d:\project_skripsi\model\model_multivariate_v3 (baru)


## Section 2 — Load Artefak Preprocessing (dari processed_v2)

In [3]:
with open(DATA_PROCESSED / "preprocessing_meta.json") as f:
    meta = json.load(f)

DAILY_FEATURES   = meta['daily_features']
MONTHLY_FEATURES = meta['monthly_features']
TARGET_COL       = meta['target_col']
TARGET_IDX       = meta['target_idx_in_daily']

print(f"Preprocessing version  : {meta['version']}")
print(f"Cutoff date            : {meta['cutoff_date']}")
print(f"Daily features         : {DAILY_FEATURES}")
print(f"Monthly features       : {MONTHLY_FEATURES}")
print(f"Target                 : {TARGET_COL}")


Preprocessing version  : v2
Cutoff date            : 2022-12-21
Daily features         : ['ihsg', 'nilai_tukar']
Monthly features       : ['inflasi', 'bi_rate']
Target                 : ihsg


In [4]:
scaler_daily   = joblib.load(DATA_PROCESSED / "scaler_daily.joblib")
scaler_monthly = joblib.load(DATA_PROCESSED / "scaler_monthly.joblib")
scaler_target  = joblib.load(DATA_PROCESSED / "scaler_target.joblib")
scaler_uni     = joblib.load(DATA_PROCESSED / "scaler_univariate.joblib")

sequences_dual = joblib.load(DATA_PROCESSED / "sequences_dual.joblib")
sequences_uni  = joblib.load(DATA_PROCESSED / "sequences_uni.joblib")

print(f"scaler_target IHSG train range: [{scaler_target.data_min_[0]:.0f}, {scaler_target.data_max_[0]:.0f}]")
print(f"\nsequences_dual: {len(sequences_dual)} kombinasi tersedia: {list(sequences_dual.keys())}")
print(f"sequences_uni : seq_daily tersedia: {list(sequences_uni.keys())}")

# Verifikasi seq_daily=90 tersedia untuk semua seq_monthly
assert 90 in sequences_uni, "seq_daily=90 tidak ada di sequences_uni!"
for sm in [3, 6, 12]:
    assert (90, sm) in sequences_dual, f"(90,{sm}) tidak ada di sequences_dual!"
print("\n✓ seq_daily=90 tersedia untuk univariate dan semua seq_monthly multivariate")


scaler_target IHSG train range: [3938, 7318]

sequences_dual: 9 kombinasi tersedia: [(22, 3), (22, 6), (22, 12), (60, 3), (60, 6), (60, 12), (90, 3), (90, 6), (90, 12)]
sequences_uni : seq_daily tersedia: [22, 60, 90]

✓ seq_daily=90 tersedia untuk univariate dan semua seq_monthly multivariate


## Section 3 — Konfigurasi Hyperparameter (Berdasarkan Observasi Empiris)

In [5]:
# ── Hyperparameter arsitektur — dikunci dari observasi empiris eksperimen sebelumnya ──
# Tidak ada grid search. Lihat tabel justifikasi di Cell 0 (markdown).
ARCH_CONFIG = {
    "seq_daily"     : 90,      # konsisten best di v1 & v2 (univariate & multivariate)
    "lstm_units"    : 128,     # konsisten lebih baik dari 64
    "dropout_rate"  : 0.2,     # standar Srivastava et al. (2014)
    "learning_rate" : 0.0001,  # terbukti lebih stabil dari 0.0005/0.001 saat retrain v1
    "batch_size"    : 32,      # aman terhadap OOM
}

# ── Ablation candidates untuk seq_monthly ──
# 3  = 1 kuartal (periode evaluasi ekonomi standar)
# 6  = setengah tahun (horizon kebijakan moneter jangka menengah)
# 12 = 1 tahun (siklus tahunan inflasi & BI rate)
SEQ_MONTHLY_CANDIDATES = [3, 6, 12]

# ── Training config ──
FINAL_EPOCHS = 200
PATIENCE_ES  = 30   # dinaikkan dari v2 (20) — beri waktu eksplorasi lebih lama
PATIENCE_LR  = 15   # dinaikkan dari v2 (10)
VAL_SPLIT    = 0.15
SEED         = 42   # reproducibility, single run

print("Arsitektur config (dikunci dari observasi empiris):")
for k, v in ARCH_CONFIG.items():
    print(f"  {k:15s}: {v}")
print(f"\nAblation seq_monthly: {SEQ_MONTHLY_CANDIDATES}")
print(f"\nTraining config:")
print(f"  FINAL_EPOCHS : {FINAL_EPOCHS}")
print(f"  PATIENCE_ES  : {PATIENCE_ES}")
print(f"  PATIENCE_LR  : {PATIENCE_LR}")
print(f"  VAL_SPLIT    : {VAL_SPLIT}")
print(f"  SEED         : {SEED}")


Arsitektur config (dikunci dari observasi empiris):
  seq_daily      : 90
  lstm_units     : 128
  dropout_rate   : 0.2
  learning_rate  : 0.0001
  batch_size     : 32

Ablation seq_monthly: [3, 6, 12]

Training config:
  FINAL_EPOCHS : 200
  PATIENCE_ES  : 30
  PATIENCE_LR  : 15
  VAL_SPLIT    : 0.15
  SEED         : 42


## Section 4 — Arsitektur Model

In [6]:
def build_lstm_dual_branch(seq_daily, seq_monthly,
                            lstm_units=128, dropout_rate=0.2,
                            learning_rate=0.0001,
                            n_daily_features=2, n_monthly_features=2):
    """
    Dual-Branch LSTM — Keras Functional API.
    Branch daily   : IHSG + Kurs (frekuensi harian)
    Branch monthly : Inflasi + BI Rate (frekuensi bulanan)
    """
    # Branch Daily
    inp_daily = Input(shape=(seq_daily, n_daily_features), name='input_daily')
    x_d = LSTM(lstm_units, return_sequences=True, name='lstm_daily_1')(inp_daily)
    x_d = Dropout(dropout_rate, name='drop_daily_1')(x_d)
    x_d = LSTM(lstm_units // 2, return_sequences=False, name='lstm_daily_2')(x_d)
    x_d = Dropout(dropout_rate, name='drop_daily_2')(x_d)

    # Branch Monthly
    inp_monthly = Input(shape=(seq_monthly, n_monthly_features), name='input_monthly')
    x_m = LSTM(lstm_units, return_sequences=True, name='lstm_monthly_1')(inp_monthly)
    x_m = Dropout(dropout_rate, name='drop_monthly_1')(x_m)
    x_m = LSTM(lstm_units // 2, return_sequences=False, name='lstm_monthly_2')(x_m)
    x_m = Dropout(dropout_rate, name='drop_monthly_2')(x_m)

    # Merge
    merged = Concatenate(name='concat')([x_d, x_m])
    x      = Dense(lstm_units // 4, activation='relu', name='dense_hidden')(merged)
    x      = Dropout(dropout_rate, name='drop_final')(x)
    output = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=[inp_daily, inp_monthly], outputs=output,
                  name='LSTM_DualBranch_Multivariate')
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='mse', metrics=['mae'])
    return model


def build_lstm_univariate(seq_daily, lstm_units=128, dropout_rate=0.2,
                           learning_rate=0.0001):
    """
    Univariate LSTM — arsitektur identik satu branch dari Dual-Branch.
    Input: IHSG only. Digunakan sebagai baseline perbandingan.
    """
    inp    = Input(shape=(seq_daily, 1), name='input_ihsg')
    x      = LSTM(lstm_units, return_sequences=True, name='lstm_1')(inp)
    x      = Dropout(dropout_rate, name='drop_1')(x)
    x      = LSTM(lstm_units // 2, return_sequences=False, name='lstm_2')(x)
    x      = Dropout(dropout_rate, name='drop_2')(x)
    x      = Dense(lstm_units // 4, activation='relu', name='dense_hidden')(x)
    x      = Dropout(dropout_rate, name='drop_final')(x)
    output = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=inp, outputs=output, name='LSTM_Univariate')
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='mse', metrics=['mae'])
    return model


def compute_metrics(y_true_scaled, y_pred_scaled, scaler_t):
    """Inverse transform dan hitung RMSE, MAPE, R² dalam skala asli IHSG."""
    y_true = scaler_t.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()
    y_pred = scaler_t.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mape   = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    r2     = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mape": mape, "r2": r2, "y_true": y_true, "y_pred": y_pred}


def make_callbacks(ckpt_path):
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=PATIENCE_LR, min_lr=1e-7, verbose=0),
        ModelCheckpoint(filepath=ckpt_path, monitor='val_loss',
                        save_best_only=True, verbose=0),
    ]


print("✓ build_lstm_dual_branch()")
print("✓ build_lstm_univariate()")
print("✓ compute_metrics()")
print("✓ make_callbacks()")


✓ build_lstm_dual_branch()
✓ build_lstm_univariate()
✓ compute_metrics()
✓ make_callbacks()


## Section 5 — Ablation Study: `seq_monthly` (Multivariate)

Satu-satunya parameter yang belum terjawab dari observasi empiris sebelumnya.
Setiap kandidat dilatih full training (seed=42) dengan `ARCH_CONFIG` yang
sudah dikunci, lalu dievaluasi di test set. Pemilihan berdasarkan RMSE terkecil.


In [7]:
print(f"Ablation study seq_monthly ∈ {SEQ_MONTHLY_CANDIDATES}")
print(f"Config arsitektur dikunci: seq_daily={ARCH_CONFIG['seq_daily']}, "
      f"lstm_units={ARCH_CONFIG['lstm_units']}, dropout={ARCH_CONFIG['dropout_rate']}, "
      f"lr={ARCH_CONFIG['learning_rate']}, batch_size={ARCH_CONFIG['batch_size']}")
print(f"Training: max {FINAL_EPOCHS} epoch, ES patience={PATIENCE_ES}, LR patience={PATIENCE_LR}")
print("=" * 70)

ablation_results = []

for seq_m in SEQ_MONTHLY_CANDIDATES:
    key  = (ARCH_CONFIG['seq_daily'], seq_m)
    seqs = sequences_dual[key]

    X_d_tr = seqs['X_daily_train']
    X_m_tr = seqs['X_monthly_train']
    y_tr   = seqs['y_train']
    X_d_te = seqs['X_daily_test']
    X_m_te = seqs['X_monthly_test']
    y_te   = seqs['y_test']

    print(f"\n[seq_monthly={seq_m}] Training...", end=" ", flush=True)

    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    tf.keras.backend.clear_session()
    gc.collect()

    ckpt_path = str(MODEL_DIR / f"ablation_multi_seqm{seq_m}.keras")
    model = build_lstm_dual_branch(
        seq_daily=ARCH_CONFIG['seq_daily'], seq_monthly=seq_m,
        lstm_units=ARCH_CONFIG['lstm_units'],
        dropout_rate=ARCH_CONFIG['dropout_rate'],
        learning_rate=ARCH_CONFIG['learning_rate'],
    )
    history = model.fit(
        [X_d_tr, X_m_tr], y_tr,
        epochs=FINAL_EPOCHS,
        batch_size=ARCH_CONFIG['batch_size'],
        validation_split=VAL_SPLIT,
        callbacks=make_callbacks(ckpt_path),
        shuffle=False, verbose=0
    )

    # Load checkpoint terbaik sebelum evaluasi
    model_eval = tf.keras.models.load_model(ckpt_path)
    y_pred = model_eval.predict([X_d_te, X_m_te], verbose=0).flatten()
    metrics = compute_metrics(y_te, y_pred, scaler_target)

    result = {
        "seq_monthly"   : seq_m,
        "best_val_loss" : float(min(history.history['val_loss'])),
        "epochs_ran"    : len(history.history['loss']),
        "rmse"          : float(metrics['rmse']),
        "mape"          : float(metrics['mape']),
        "r2"            : float(metrics['r2']),
        "checkpoint"    : ckpt_path,
    }
    ablation_results.append(result)
    print(f"epochs={result['epochs_ran']:3d} | val_loss={result['best_val_loss']:.6f} "
          f"| RMSE={result['rmse']:.2f} MAPE={result['mape']:.4f}% R²={result['r2']:.4f}")

print()
print("=" * 70)
print("ABLATION RESULTS — seq_monthly:")
print(f"  {'seq_m':>6} {'epochs':>7} {'val_loss':>10} {'RMSE':>8} {'MAPE':>8} {'R²':>8}")
print(f"  {'-'*52}")
for r in ablation_results:
    print(f"  {r['seq_monthly']:>6} {r['epochs_ran']:>7} {r['best_val_loss']:>10.6f} "
          f"{r['rmse']:>8.2f} {r['mape']:>8.4f} {r['r2']:>8.4f}")

best_ablation    = min(ablation_results, key=lambda x: x['rmse'])
best_seq_monthly = best_ablation['seq_monthly']
print(f"\n✓ Best seq_monthly: {best_seq_monthly} "
      f"(RMSE={best_ablation['rmse']:.2f}, R²={best_ablation['r2']:.4f})")

with open(MODEL_DIR / "ablation_seq_monthly.json", "w") as f:
    json.dump({
        "arch_config_fixed" : ARCH_CONFIG,
        "candidates"        : SEQ_MONTHLY_CANDIDATES,
        "results"           : ablation_results,
        "best_seq_monthly"  : best_seq_monthly,
        "selection_metric"  : "rmse (terkecil)",
    }, f, indent=2)
print(f"✓ Disimpan → ablation_seq_monthly.json")


Ablation study seq_monthly ∈ [3, 6, 12]
Config arsitektur dikunci: seq_daily=90, lstm_units=128, dropout=0.2, lr=0.0001, batch_size=32
Training: max 200 epoch, ES patience=30, LR patience=15

[seq_monthly=3] Training... WARNING:tensorflow:From d:\project_skripsi\.venv\lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.

epochs= 66 | val_loss=0.001448 | RMSE=126.37 MAPE=1.3433% R²=0.8170

[seq_monthly=6] Training... epochs= 75 | val_loss=0.001544 | RMSE=215.27 MAPE=2.5286% R²=0.4689

[seq_monthly=12] Training... epochs=106 | val_loss=0.002086 | RMSE=159.23 MAPE=1.7764% R²=0.7094

ABLATION RESULTS — seq_monthly:
   seq_m  epochs   val_loss     RMSE     MAPE       R²
  ----------------------------------------------------
       3      66   0.001448   126.37   1.3433   0.8170
       6      75   0.001544   215.27   2.5286   0.4689
      12     106   0.002086   159.23   1.7764   0.7

## Section 6 — Konfigurasi & Data Final

In [8]:
# ── Final config multivariate ──
FINAL_CONFIG_MULTI = dict(ARCH_CONFIG)
FINAL_CONFIG_MULTI['seq_monthly'] = best_seq_monthly

key_m        = (FINAL_CONFIG_MULTI['seq_daily'], FINAL_CONFIG_MULTI['seq_monthly'])
seqs_m       = sequences_dual[key_m]
X_d_tr_final = seqs_m['X_daily_train']
X_m_tr_final = seqs_m['X_monthly_train']
y_tr_final   = seqs_m['y_train']
X_d_te_final = seqs_m['X_daily_test']
X_m_te_final = seqs_m['X_monthly_test']
y_te_final   = seqs_m['y_test']

print("Final multivariate config:")
for k, v in FINAL_CONFIG_MULTI.items():
    print(f"  {k:15s}: {v}")
print(f"\nTrain: X_daily={X_d_tr_final.shape}, X_monthly={X_m_tr_final.shape}, y={y_tr_final.shape}")
print(f"Test : X_daily={X_d_te_final.shape}, X_monthly={X_m_te_final.shape}, y={y_te_final.shape}")
print(f"y_te range: [{y_te_final.min():.4f}, {y_te_final.max():.4f}]")

# ── Final config univariate ──
FINAL_CONFIG_UNI = dict(ARCH_CONFIG)  # tidak ada seq_monthly

X_tr_uni_final = sequences_uni[FINAL_CONFIG_UNI['seq_daily']]['X_train']
y_tr_uni_final = sequences_uni[FINAL_CONFIG_UNI['seq_daily']]['y_train']
X_te_uni_final = sequences_uni[FINAL_CONFIG_UNI['seq_daily']]['X_test']
y_te_uni_final = sequences_uni[FINAL_CONFIG_UNI['seq_daily']]['y_test']

print(f"\nFinal univariate config:")
for k, v in FINAL_CONFIG_UNI.items():
    print(f"  {k:15s}: {v}")
print(f"\nTrain: X={X_tr_uni_final.shape}, y={y_tr_uni_final.shape}")
print(f"Test : X={X_te_uni_final.shape}, y={y_te_uni_final.shape}")
print(f"y_te range: [{y_te_uni_final.min():.4f}, {y_te_uni_final.max():.4f}]")


Final multivariate config:
  seq_daily      : 90
  lstm_units     : 128
  dropout_rate   : 0.2
  learning_rate  : 0.0001
  batch_size     : 32
  seq_monthly    : 3

Train: X_daily=(1836, 90, 2), X_monthly=(1836, 3, 2), y=(1836,)
Test : X_daily=(485, 90, 2), X_monthly=(485, 3, 2), y=(485,)
y_te range: [0.7775, 1.1738]

Final univariate config:
  seq_daily      : 90
  lstm_units     : 128
  dropout_rate   : 0.2
  learning_rate  : 0.0001
  batch_size     : 32

Train: X=(1849, 90, 1), y=(1849,)
Test : X=(485, 90, 1), y=(485,)
y_te range: [0.7775, 1.1738]


## Section 7 — Final Training Multivariate (Single Run)

Single run dengan `seed=42`, config lengkap dari `FINAL_CONFIG_MULTI`.


In [9]:
print(f"Final training multivariate — single run, seed={SEED}, max {FINAL_EPOCHS} epoch")
print(f"Config: {FINAL_CONFIG_MULTI}")
print("=" * 70)

tf.random.set_seed(SEED)
np.random.seed(SEED)
tf.keras.backend.clear_session()
gc.collect()

ckpt_path_multi = str(MODEL_DIR / "best_multi_checkpoint.keras")
model_multi = build_lstm_dual_branch(
    seq_daily=FINAL_CONFIG_MULTI['seq_daily'],
    seq_monthly=FINAL_CONFIG_MULTI['seq_monthly'],
    lstm_units=FINAL_CONFIG_MULTI['lstm_units'],
    dropout_rate=FINAL_CONFIG_MULTI['dropout_rate'],
    learning_rate=FINAL_CONFIG_MULTI['learning_rate'],
)
model_multi.summary()

history_multi = model_multi.fit(
    [X_d_tr_final, X_m_tr_final], y_tr_final,
    epochs=FINAL_EPOCHS,
    batch_size=FINAL_CONFIG_MULTI['batch_size'],
    validation_split=VAL_SPLIT,
    callbacks=make_callbacks(ckpt_path_multi),
    shuffle=False, verbose=1
)

epochs_ran_multi = len(history_multi.history['loss'])
best_val_multi   = min(history_multi.history['val_loss'])
print(f"\n✓ Training selesai — {epochs_ran_multi} epoch aktual")
print(f"  Best val_loss : {best_val_multi:.6f}")


Final training multivariate — single run, seed=42, max 200 epoch
Config: {'seq_daily': 90, 'lstm_units': 128, 'dropout_rate': 0.2, 'learning_rate': 0.0001, 'batch_size': 32, 'seq_monthly': 3}


Model: "LSTM_DualBranch_Multivariate"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_daily         │ (None, 90, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_monthly       │ (None, 3, 2)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_daily_1 (LSTM) │ (None, 90, 128)   │     67,072 │ input_daily[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_monthly_1      │ (None, 3, 128)    │     67,072 │ input_monthly[0]… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_daily_1        │ (None, 90, 128)   │          0 │ lstm_daily_1[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_monthly_1      │ (None, 3, 128)    │          0 │ lstm_monthly_1[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_daily_2 (LSTM) │ (None, 64)        │     49,408 │ drop_daily_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_monthly_2      │ (None, 64)        │     49,408 │ drop_monthly_1[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_daily_2        │ (None, 64)        │          0 │ lstm_daily_2[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_monthly_2      │ (None, 64)        │          0 │ lstm_monthly_2[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat              │ (None, 128)       │          0 │ drop_daily_2[0][… │
│ (Concatenate)       │                   │            │ drop_monthly_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_hidden        │ (None, 32)        │      4,128 │ concat[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_final          │ (None, 32)        │          0 │ dense_hidden[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         33 │ drop_final[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 237,121 (926.25 KB)

 Trainable params: 237,121 (926.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 152ms/step - loss: 0.0778 - mae: 0.2427 - val_loss: 0.0259 - val_mae: 0.1546 - learning_rate: 1.0000e-04
Epoch 2/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 140ms/step - loss: 0.0269 - mae: 0.1295 - val_loss: 0.0393 - val_mae: 0.1937 - learning_rate: 1.0000e-04
Epoch 3/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 131ms/step - loss: 0.0203 - mae: 0.1108 - val_loss: 0.0352 - val_mae: 0.1829 - learning_rate: 1.0000e-04
Epoch 4/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 131ms/step - loss: 0.0155 - mae: 0.0961 - val_loss: 0.0209 - val_mae: 0.1383 - learning_rate: 1.0000e-04
Epoch 5/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 135ms/step - loss: 0.0146 - mae: 0.0921 - val_loss: 0.0110 - val_mae: 0.0977 - learning_rate: 1.0000e-04
Epoch 6/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 140ms/step - loss: 0.0136 - mae: 0.0887 - val_loss: 0.0093 - val_mae: 0.0891 - learning_rate: 1.0000e-04
Epoch 7/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 140ms/step - loss: 0.0121 - mae: 0.0827 - val_loss: 0.0051 - val_mae: 0.065

In [10]:
# Load checkpoint terbaik
tf.keras.backend.clear_session()
gc.collect()
model_multi = tf.keras.models.load_model(ckpt_path_multi)
print(f"✓ Model multivariate loaded dari {ckpt_path_multi}")


✓ Model multivariate loaded dari d:\project_skripsi\model\model_multivariate_v3\best_multi_checkpoint.keras


## Section 8 — Final Training Univariate (Single Run)

Single run dengan `seed=42`, config dari `FINAL_CONFIG_UNI` (identik dengan
arsitektur multivariate per-branch, hanya input 1 fitur).


In [11]:
print(f"Final training univariate — single run, seed={SEED}, max {FINAL_EPOCHS} epoch")
print(f"Config: {FINAL_CONFIG_UNI}")
print("=" * 70)

tf.random.set_seed(SEED)
np.random.seed(SEED)
tf.keras.backend.clear_session()
gc.collect()

ckpt_path_uni = str(MODEL_DIR / "best_uni_checkpoint.keras")
model_uni = build_lstm_univariate(
    seq_daily=FINAL_CONFIG_UNI['seq_daily'],
    lstm_units=FINAL_CONFIG_UNI['lstm_units'],
    dropout_rate=FINAL_CONFIG_UNI['dropout_rate'],
    learning_rate=FINAL_CONFIG_UNI['learning_rate'],
)
model_uni.summary()

history_uni = model_uni.fit(
    X_tr_uni_final, y_tr_uni_final,
    epochs=FINAL_EPOCHS,
    batch_size=FINAL_CONFIG_UNI['batch_size'],
    validation_split=VAL_SPLIT,
    callbacks=make_callbacks(ckpt_path_uni),
    shuffle=False, verbose=1
)

epochs_ran_uni = len(history_uni.history['loss'])
best_val_uni   = min(history_uni.history['val_loss'])
print(f"\n✓ Training selesai — {epochs_ran_uni} epoch aktual")
print(f"  Best val_loss : {best_val_uni:.6f}")


Final training univariate — single run, seed=42, max 200 epoch
Config: {'seq_daily': 90, 'lstm_units': 128, 'dropout_rate': 0.2, 'learning_rate': 0.0001, 'batch_size': 32}


Model: "LSTM_Univariate"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_ihsg (InputLayer)         │ (None, 90, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 90, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ (None, 90, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_2 (Dropout)                │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_final (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,081 (461.25 KB)

 Trainable params: 118,081 (461.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 144ms/step - loss: 0.1038 - mae: 0.2825 - val_loss: 0.0018 - val_mae: 0.0330 - learning_rate: 1.0000e-04
Epoch 2/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 125ms/step - loss: 0.0191 - mae: 0.1103 - val_loss: 0.0059 - val_mae: 0.0687 - learning_rate: 1.0000e-04
Epoch 3/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 120ms/step - loss: 0.0170 - mae: 0.1009 - val_loss: 0.0063 - val_mae: 0.0714 - learning_rate: 1.0000e-04
Epoch 4/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 7s 131ms/step - loss: 0.0158 - mae: 0.0959 - val_loss: 0.0055 - val_mae: 0.0661 - learning_rate: 1.0000e-04
Epoch 5/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 128ms/step - loss: 0.0139 - mae: 0.0907 - val_loss: 0.0052 - val_mae: 0.0647 - learning_rate: 1.0000e-04
Epoch 6/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 125ms/step - loss: 0.0132 - mae: 0.0883 - val_loss: 0.0025 - val_mae: 0.0424 - learning_rate: 1.0000e-04
Epoch 7/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 120ms/step - loss: 0.0135 - mae: 0.0892 - val_loss: 0.0027 - val_mae: 0.045

In [12]:
tf.keras.backend.clear_session()
gc.collect()
model_uni = tf.keras.models.load_model(ckpt_path_uni)
print(f"✓ Model univariate loaded dari {ckpt_path_uni}")


✓ Model univariate loaded dari d:\project_skripsi\model\model_multivariate_v3\best_uni_checkpoint.keras


## Section 9 — Evaluasi Test Set

In [13]:
y_pred_multi_scaled = model_multi.predict([X_d_te_final, X_m_te_final], verbose=0).flatten()
y_pred_uni_scaled   = model_uni.predict(X_te_uni_final, verbose=0).flatten()

print("Diagnostik skala prediksi:")
print(f"  y_te_final (multi) : [{y_te_final.min():.4f}, {y_te_final.max():.4f}]  mean={y_te_final.mean():.4f}")
print(f"  pred_multi         : [{y_pred_multi_scaled.min():.4f}, {y_pred_multi_scaled.max():.4f}]  mean={y_pred_multi_scaled.mean():.4f}")
print(f"  y_te_uni_final     : [{y_te_uni_final.min():.4f}, {y_te_uni_final.max():.4f}]  mean={y_te_uni_final.mean():.4f}")
print(f"  pred_uni           : [{y_pred_uni_scaled.min():.4f}, {y_pred_uni_scaled.max():.4f}]  mean={y_pred_uni_scaled.mean():.4f}")

metrics_multi = compute_metrics(y_te_final,     y_pred_multi_scaled, scaler_target)
metrics_uni   = compute_metrics(y_te_uni_final, y_pred_uni_scaled,   scaler_uni)

print("\n" + "=" * 55)
print(f"{'Metrik':<12} {'Univariate':>15} {'Multivariate':>15}")
print("-" * 55)
print(f"{'RMSE':<12} {metrics_uni['rmse']:>15.2f} {metrics_multi['rmse']:>15.2f}")
print(f"{'MAPE (%)':<12} {metrics_uni['mape']:>15.4f} {metrics_multi['mape']:>15.4f}")
print(f"{'R²':<12} {metrics_uni['r2']:>15.4f} {metrics_multi['r2']:>15.4f}")
print("=" * 55)

delta_rmse = metrics_uni['rmse'] - metrics_multi['rmse']
delta_mape = metrics_uni['mape'] - metrics_multi['mape']
delta_r2   = metrics_multi['r2'] - metrics_uni['r2']
winner     = "multivariate" if delta_rmse > 0 else "univariate"

print(f"\nDelta (Multi − Uni):")
print(f"  RMSE : {delta_rmse:+.2f}  ({'Multivariate lebih baik' if delta_rmse > 0 else 'Univariate lebih baik'})")
print(f"  MAPE : {delta_mape:+.4f}% ({'Multivariate lebih baik' if delta_mape > 0 else 'Univariate lebih baik'})")
print(f"  R²   : {delta_r2:+.4f}  ({'Multivariate lebih baik' if delta_r2 > 0 else 'Univariate lebih baik'})")
print(f"\n→ Winner: {winner.upper()}")


Diagnostik skala prediksi:
  y_te_final (multi) : [0.7775, 1.1738]  mean=0.9297
  pred_multi         : [0.7753, 1.1337]  mean=0.9190
  y_te_uni_final     : [0.7775, 1.1738]  mean=0.9297
  pred_uni           : [0.8122, 1.1333]  mean=0.9292

Metrik            Univariate    Multivariate
-------------------------------------------------------
RMSE                  129.05          122.89
MAPE (%)              1.4227          1.3466
R²                    0.8091          0.8269

Delta (Multi − Uni):
  RMSE : +6.17  (Multivariate lebih baik)
  MAPE : +0.0761% (Multivariate lebih baik)
  R²   : +0.0178  (Multivariate lebih baik)

→ Winner: MULTIVARIATE


## Section 10 — Simpan Artefak

In [14]:
model_multi.save(MODEL_DIR / "model_multivariate.keras")
model_uni.save(MODEL_DIR / "model_univariate.keras")
print("✓ model_multivariate.keras")
print("✓ model_univariate.keras")

model_config = {
    "created_at"  : datetime.now().isoformat(),
    "version"     : "v3",
    "preprocessing_version": "v2 (reused)",
    "approach"    : "single run, hyperparameter dari observasi empiris (tidak ada grid search)",
    "notes"       : [
        "Hyperparameter arsitektur dikunci dari observasi empiris eksperimen v1 & v2",
        "seq_monthly dipilih melalui ablation study full training",
        "Final training single run dengan seed=42, patience ES=30 LR=15",
        "y_test konsisten (canonical) dari preprocessing v2"
    ],
    "arch_config_fixed": ARCH_CONFIG,
    "multivariate": {
        "architecture"         : "Dual-Branch LSTM (Keras Functional API)",
        "config"               : FINAL_CONFIG_MULTI,
        "seq_monthly_selection": "ablation study (RMSE terkecil)",
        "ablation_results"     : ablation_results,
        "epochs_ran"           : epochs_ran_multi,
        "best_val_loss"        : best_val_multi,
        "test_metrics"         : {
            "rmse": round(float(metrics_multi['rmse']), 4),
            "mape": round(float(metrics_multi['mape']), 4),
            "r2"  : round(float(metrics_multi['r2']),   4),
        },
        "input_daily_shape"    : list(X_d_te_final.shape[1:]),
        "input_monthly_shape"  : list(X_m_te_final.shape[1:]),
        "daily_features"       : DAILY_FEATURES,
        "monthly_features"     : MONTHLY_FEATURES,
        "target_col"           : TARGET_COL,
    },
    "univariate"  : {
        "architecture" : "Single-Branch LSTM",
        "config"       : FINAL_CONFIG_UNI,
        "epochs_ran"   : epochs_ran_uni,
        "best_val_loss": best_val_uni,
        "test_metrics" : {
            "rmse": round(float(metrics_uni['rmse']), 4),
            "mape": round(float(metrics_uni['mape']), 4),
            "r2"  : round(float(metrics_uni['r2']),   4),
        },
        "input_shape"  : list(X_te_uni_final.shape[1:]),
        "target_col"   : TARGET_COL,
    },
    "comparison"  : {
        "delta_rmse": round(float(delta_rmse), 4),
        "delta_mape": round(float(delta_mape), 4),
        "delta_r2"  : round(float(delta_r2),   4),
        "winner"    : winner,
    }
}

with open(MODEL_DIR / "model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)
print("✓ model_config.json")

# Training history (untuk plot loss curve)
history_data = {
    "multivariate": {
        "loss"    : [float(x) for x in history_multi.history['loss']],
        "val_loss": [float(x) for x in history_multi.history['val_loss']],
    },
    "univariate": {
        "loss"    : [float(x) for x in history_uni.history['loss']],
        "val_loss": [float(x) for x in history_uni.history['val_loss']],
    },
}
with open(MODEL_DIR / "training_history.json", "w") as f:
    json.dump(history_data, f, indent=2)
print("✓ training_history.json")

# Predictions untuk visualisasi
dates_test_multi = sequences_dual[key_m]['dates_test']
dates_test_uni   = sequences_uni[FINAL_CONFIG_UNI['seq_daily']].get(
    'dates_test', [None] * len(y_te_uni_final)
)

predictions_data = {
    "multivariate": {
        "dates" : [str(pd.Timestamp(d).date()) if d is not None else None
                   for d in dates_test_multi],
        "y_true": metrics_multi['y_true'].tolist(),
        "y_pred": metrics_multi['y_pred'].tolist(),
    },
    "univariate"  : {
        "dates" : [str(pd.Timestamp(d).date()) if d is not None else None
                   for d in dates_test_uni],
        "y_true": metrics_uni['y_true'].tolist(),
        "y_pred": metrics_uni['y_pred'].tolist(),
    },
}
joblib.dump(predictions_data, MODEL_DIR / "predictions_test.joblib")
print("✓ predictions_test.joblib")

print()
print("=" * 60)
print(f"File tersimpan di: {MODEL_DIR}")
print("=" * 60)
for p in sorted(MODEL_DIR.iterdir()):
    print(f"  {p.name:45s} {p.stat().st_size/1024:8.1f} KB")


✓ model_multivariate.keras
✓ model_univariate.keras
✓ model_config.json
✓ training_history.json
✓ predictions_test.joblib

File tersimpan di: d:\project_skripsi\model\model_multivariate_v3
  ablation_multi_seqm12.keras                     2851.7 KB
  ablation_multi_seqm3.keras                      2851.7 KB
  ablation_multi_seqm6.keras                      2851.7 KB
  ablation_seq_monthly.json                          1.2 KB
  best_multi_checkpoint.keras                     2851.7 KB
  best_uni_checkpoint.keras                       1430.0 KB
  model_config.json                                  3.0 KB
  model_multivariate.keras                        2851.3 KB
  model_univariate.keras                          1429.7 KB
  predictions_test.joblib                           23.8 KB
  training_history.json                              6.5 KB


## Section 11 — Final Summary

In [15]:
with open(MODEL_DIR / "model_config.json") as f:
    cfg = json.load(f)

print("=" * 60)
print("FINAL SUMMARY — MODELING v3 (Single Run, Empirical Config)")
print("=" * 60)
print(f"  Version              : {cfg['version']}")
print(f"  Approach             : {cfg['approach']}")
print()
print(f"  ── Multivariate ──────────────────────────────")
c = cfg['multivariate']['config']
print(f"  seq_daily            : {c['seq_daily']}")
print(f"  seq_monthly          : {c['seq_monthly']}  (dari ablation study)")
print(f"  lstm_units           : {c['lstm_units']}")
print(f"  dropout_rate         : {c['dropout_rate']}")
print(f"  learning_rate        : {c['learning_rate']}")
print(f"  epochs_ran           : {cfg['multivariate']['epochs_ran']}")
print(f"  RMSE                 : {cfg['multivariate']['test_metrics']['rmse']}")
print(f"  MAPE                 : {cfg['multivariate']['test_metrics']['mape']}%")
print(f"  R²                   : {cfg['multivariate']['test_metrics']['r2']}")
print()
print(f"  ── Univariate ────────────────────────────────")
c_u = cfg['univariate']['config']
print(f"  seq_daily            : {c_u['seq_daily']}")
print(f"  lstm_units           : {c_u['lstm_units']}")
print(f"  learning_rate        : {c_u['learning_rate']}")
print(f"  epochs_ran           : {cfg['univariate']['epochs_ran']}")
print(f"  RMSE                 : {cfg['univariate']['test_metrics']['rmse']}")
print(f"  MAPE                 : {cfg['univariate']['test_metrics']['mape']}%")
print(f"  R²                   : {cfg['univariate']['test_metrics']['r2']}")
print()
print(f"  ── Perbandingan ──────────────────────────────")
print(f"  ΔRMSE (Multi−Uni)    : {cfg['comparison']['delta_rmse']:+.4f}")
print(f"  ΔR²   (Multi−Uni)    : {cfg['comparison']['delta_r2']:+.4f}")
print(f"  Winner               : {cfg['comparison']['winner'].upper()}")
print("=" * 60)


FINAL SUMMARY — MODELING v3 (Single Run, Empirical Config)
  Version              : v3
  Approach             : single run, hyperparameter dari observasi empiris (tidak ada grid search)

  ── Multivariate ──────────────────────────────
  seq_daily            : 90
  seq_monthly          : 3  (dari ablation study)
  lstm_units           : 128
  dropout_rate         : 0.2
  learning_rate        : 0.0001
  epochs_ran           : 70
  RMSE                 : 122.8858
  MAPE                 : 1.3466%
  R²                   : 0.8269

  ── Univariate ────────────────────────────────
  seq_daily            : 90
  lstm_units           : 128
  learning_rate        : 0.0001
  epochs_ran           : 41
  RMSE                 : 129.0521
  MAPE                 : 1.4227%
  R²                   : 0.8091

  ── Perbandingan ──────────────────────────────
  ΔRMSE (Multi−Uni)    : +6.1663
  ΔR²   (Multi−Uni)    : +0.0178
  Winner               : MULTIVARIATE
